# EXP-T + EXP-V — text and image TTFT curves

Two sweeps on one server (Qwen2-VL-2B, fp16):
- **EXP-T**: unique text prompts at 64 / 256 / 576 / 1024 tokens, 10 runs each
- **EXP-V**: unique noise images at 224 / 448 / 672 / 896 px (same token counts), 10 runs each

Warmup requests run first (excluded from data) so one-time kernel compilation doesn't pollute the curves.
Every request appends one line to `/content/results_exp_tv.jsonl` — a dropped session loses nothing.

Runtime → Change runtime type → **T4 GPU**. Total runtime ≈ 35–45 min.


In [ ]:
!nvidia-smi -L


In [ ]:
# Install vLLM 0.25.1 built for CUDA 12.9 (the PyPI default targets CUDA 13 and fails on Colab)
%pip -q install https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl --extra-index-url https://download.pytorch.org/whl/cu129
import vllm, transformers
print('vllm', vllm.__version__, '| transformers', transformers.__version__)


In [ ]:
# Launch the server (same verified flags as EXP-0)
import subprocess
args = [
    'vllm', 'serve', 'Qwen/Qwen2-VL-2B-Instruct',
    '--dtype', 'half', '--max-model-len', '4096',
    '--gpu-memory-utilization', '0.85', '--enforce-eager',
    '--limit-mm-per-prompt', '{"image": 1}',
    '--mm-processor-kwargs', '{"min_pixels": 784, "max_pixels": 1003520}',
    '--port', '8000',
]
log_file = open('/content/vllm.log', 'w')
server = subprocess.Popen(args, stdout=log_file, stderr=subprocess.STDOUT)
print('server starting, pid', server.pid)


In [ ]:
# Wait for readiness
import time, requests
for attempt in range(120):
    if server.poll() is not None:
        print(open('/content/vllm.log').read()[-3000:])
        raise RuntimeError('server exited - log above')
    try:
        if requests.get('http://localhost:8000/v1/models', timeout=2).ok:
            print(f'server ready after ~{attempt*5}s'); break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    raise RuntimeError('server not ready after 10 min')


## Benchmark code (mirrors `bench/` in the repo)


In [ ]:
import base64, io, json, time
import numpy as np
import requests
from PIL import Image
from transformers import AutoTokenizer

MODEL = 'Qwen/Qwen2-VL-2B-Instruct'
RESULTS = '/content/results_exp_tv.jsonl'

import base64
import io

import numpy as np
from PIL import Image


def image_tokens(width: int, height: int) -> int:
    """Tokens an image costs in Qwen2-VL: one per 28x28 pixel block."""
    return (height // 28) * (width // 28)


def noise_image(side_px: int, seed: int) -> Image.Image:
    """A square RGB random-noise image. Noise (not a flat color) so the JPEG
    is realistically sized; a distinct seed gives a distinct image, which
    defeats prefix caching in cold experiments."""
    if side_px % 28 != 0:
        raise ValueError(f"side_px must be a multiple of 28, got {side_px}")
    rng = np.random.default_rng(seed)
    pixels = rng.integers(0, 256, size=(side_px, side_px, 3), dtype=np.uint8)
    return Image.fromarray(pixels)


def to_data_url(img: Image.Image) -> str:
    """Encode an image as the data-URL string the OpenAI-style API expects."""
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{b64}"


def text_prompt(target_tokens: int, tokenizer, seed: int) -> str:
    """A unique string that tokenizes to exactly target_tokens tokens
    (measured without special tokens; the chat template adds its own).

    Build over-long text, trim the token ids, decode back — then repair,
    because decode/encode is not a perfect round trip and the count can
    shift by a token."""
    text = f"case {seed}:" + " benchmark" * target_tokens
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) < target_tokens:
        raise ValueError(f"filler too short for {target_tokens} tokens")
    text = tokenizer.decode(ids[:target_tokens])

    for _ in range(50):
        count = len(tokenizer.encode(text, add_special_tokens=False))
        if count == target_tokens:
            return text
        if count > target_tokens:
            ids = tokenizer.encode(text, add_special_tokens=False)[:-1]
            text = tokenizer.decode(ids)
        else:
            text += " ok"
    raise RuntimeError(f"could not converge on {target_tokens} tokens")


def text_payload(model: str, prompt: str, max_tokens: int = 64) -> dict:
    """Request body for a text-only chat completion. temperature=0 so the
    same input always gives the same output (reproducible benchmarks)."""
    return {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "max_tokens": max_tokens,
    }


def image_payload(model: str, data_url: str, question: str, max_tokens: int = 64) -> dict:
    """Request body for an image + question chat completion. The content is a
    list of blocks: one text block, one image block (the base64 data URL)."""
    return {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            }
        ],
        "temperature": 0,
        "max_tokens": max_tokens,
    }

import json
import time

def send_request(payload: dict, base_url: str = "http://localhost:8000") -> dict:
    """POST one chat completion and stream the reply. Returns timings,
    the server-reported prompt token count, and the full output text."""
    payload = dict(payload, stream=True, stream_options={"include_usage": True})

    t_start = time.perf_counter()
    ttft = None
    parts = []
    usage = None

    url = f"{base_url}/v1/chat/completions"
    with requests.post(url, json=payload, stream=True, timeout=600) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line.startswith(b"data: "):
                continue
            data = line[len(b"data: "):]
            if data == b"[DONE]":
                break
            chunk = json.loads(data)
            if chunk.get("usage"):
                usage = chunk["usage"]
            for choice in chunk.get("choices", []):
                content = choice.get("delta", {}).get("content")
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - t_start
                    parts.append(content)

    return {
        "ttft_s": ttft,
        "e2e_s": time.perf_counter() - t_start,
        "prompt_tokens": usage["prompt_tokens"] if usage else None,
        "completion_tokens": usage["completion_tokens"] if usage else None,
        "output_text": "".join(parts),
    }

import json



def append_jsonl(path: str, record: dict) -> None:
    with open(path, "a") as f:
        f.write(json.dumps(record) + "\n")


def warmup(model: str, tokenizer, base_url: str = "http://localhost:8000") -> None:
    """Throwaway requests so one-time costs (Triton kernel compilation,
    per-shape vision kernels) are paid before any measured request."""
    print("warmup: text")
    send_request(text_payload(model, text_prompt(64, tokenizer, seed=999_000), max_tokens=8), base_url)
    for side in (224, 448, 672, 896):
        print(f"warmup: image {side}px")
        payload = image_payload(model, to_data_url(noise_image(side, seed=999_000 + side)), "Describe.", max_tokens=8)
        send_request(payload, base_url)


def run_text_sweep(model, tokenizer, targets, runs_per_target, out_path,
                   base_url: str = "http://localhost:8000") -> None:
    """EXP-T: unique text prompts at each target token count."""
    seed = 0
    for target in targets:
        for run in range(runs_per_target):
            seed += 1
            prompt = text_prompt(target, tokenizer, seed)
            result = send_request(text_payload(model, prompt, max_tokens=16), base_url)
            append_jsonl(out_path, {
                "experiment": "EXP-T",
                "target_tokens": target,
                "seed": seed,
                "prompt_tokens": result["prompt_tokens"],
                "ttft_s": result["ttft_s"],
                "e2e_s": result["e2e_s"],
            })
            print(f"EXP-T {target:>4} tok  run {run + 1}/{runs_per_target}  "
                  f"prompt_tokens={result['prompt_tokens']}  ttft={result['ttft_s']:.3f}s")


def run_image_sweep(model, sides, runs_per_side, out_path,
                    base_url: str = "http://localhost:8000") -> None:
    """EXP-V: unique noise images at each resolution."""
    seed = 0
    for side in sides:
        for run in range(runs_per_side):
            seed += 1
            data_url = to_data_url(noise_image(side, seed))
            payload = image_payload(model, data_url, "Describe this image.", max_tokens=16)
            result = send_request(payload, base_url)
            append_jsonl(out_path, {
                "experiment": "EXP-V",
                "side_px": side,
                "seed": seed,
                "prompt_tokens": result["prompt_tokens"],
                "ttft_s": result["ttft_s"],
                "e2e_s": result["e2e_s"],
            })
            print(f"EXP-V {side:>3}px  run {run + 1}/{runs_per_side}  "
                  f"prompt_tokens={result['prompt_tokens']}  ttft={result['ttft_s']:.3f}s")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
print('benchmark code loaded')


In [ ]:
# Warmup (excluded from data): pays one-time kernel compilation per shape
warmup(MODEL, tokenizer)
print('warmup done')


In [ ]:
# EXP-T: text sweep (~40 requests, a few minutes)
run_text_sweep(MODEL, tokenizer, targets=[64, 256, 576, 1024], runs_per_target=10, out_path=RESULTS)


In [ ]:
# EXP-V: image sweep (~40 requests)
run_image_sweep(MODEL, sides=[224, 448, 672, 896], runs_per_side=10, out_path=RESULTS)


## Analysis: fits and the overlay plot


In [ ]:
import numpy as np

import json

import numpy as np


def load_jsonl(path: str) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def fit_line(records: list[dict]) -> tuple[float, float]:
    """Least-squares TTFT vs prompt_tokens.
    Returns (slope in ms per token, intercept in ms)."""
    x = np.array([r["prompt_tokens"] for r in records], dtype=float)
    y = np.array([r["ttft_s"] for r in records], dtype=float) * 1000.0
    slope, intercept = np.polyfit(x, y, 1)
    return float(slope), float(intercept)


def median_ttft_by(records: list[dict], key: str) -> dict:
    """Median TTFT (seconds) per group, e.g. per target_tokens or side_px."""
    groups: dict = {}
    for r in records:
        groups.setdefault(r[key], []).append(r["ttft_s"])
    return {k: float(np.median(v)) for k, v in sorted(groups.items())}

records = load_jsonl(RESULTS)
text_recs = [r for r in records if r['experiment'] == 'EXP-T']
img_recs = [r for r in records if r['experiment'] == 'EXP-V']

t_slope, t_int = fit_line(text_recs)
i_slope, i_int = fit_line(img_recs)

print(f'TEXT : ttft_ms = {t_int:8.1f} + {t_slope:.4f} * prompt_tokens')
print(f'IMAGE: ttft_ms = {i_int:8.1f} + {i_slope:.4f} * prompt_tokens')
print()
print(f'per-100-token prefill cost: text {t_slope*100:.1f} ms | image {i_slope*100:.1f} ms')
print(f'fixed encoder overhead (intercept gap): {i_int - t_int:.1f} ms')
print()
print('median TTFT by point:')
print('  text :', {k: round(v, 3) for k, v in median_ttft_by(text_recs, 'target_tokens').items()})
print('  image:', {k: round(v, 3) for k, v in median_ttft_by(img_recs, 'side_px').items()})


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for recs, color, label in ((text_recs, '#3056C9', 'text-only prompt'), (img_recs, '#C2410C', 'image prompt')):
    x = [r['prompt_tokens'] for r in recs]
    y = [r['ttft_s'] * 1000 for r in recs]
    ax.scatter(x, y, s=18, color=color, alpha=0.55, label=label)
    slope, intercept = fit_line(recs)
    xs = np.array([min(x), max(x)])
    ax.plot(xs, intercept + slope * xs, color=color, linewidth=2)
ax.set_xlabel('prompt tokens (server-reported)')
ax.set_ylabel('TTFT (ms)')
ax.set_title('TTFT vs prompt tokens - Qwen2-VL-2B, vLLM 0.25.1, T4 fp16')
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.savefig('/content/ttft_overlay.png', dpi=150, bbox_inches='tight')
plt.show()


## Done

1. Copy the analysis-cell output back to the Claude session.
2. Download `results_exp_tv.jsonl` and `ttft_overlay.png` (Files panel on the left) and drop them into the repo's `results/` folder.
